In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Hp\\Desktop\\End-to-End-Machine_learning_project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Hp\\Desktop\\End-to-End-Machine_learning_project'

Entity

In [ ]:
from dataclasses import dataclass
from pathlib import Path



@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    # params
    alpha: float 
    l1_ratio: float
    # schema
    target_column: str

update the configuration manger is src config

In [6]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_model_trainer_config(self)-> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN


        create_directories([config.root_dir])
        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            model_name=config.model_name,
            alpha= params.alpha,
            l1_ratio=params.l1_ratio,
            target_column=schema.name


        )


        return model_trainer_config


component

In [8]:
import pandas as pd
import os
import logging
from sklearn.linear_model import ElasticNet
import joblib

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


In [9]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config


    def train(self):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)


        train_x =  train_data.drop([self.config.target_column], axis=1)
        test_x = test_data.drop([self.config.target_column], axis=1)
        train_y = train_data[[self.config.target_column]]
        test_y = test_data[[self.config.target_column]]


        lr = ElasticNet(alpha=self.config.alpha,
                        l1_ratio=self.config.l1_ratio,
                        random_state=42)
        lr.fit(train_x, train_y)

        joblib.dump(lr, os.path.join(self.config.root_dir, self.config.model_name))








pipeline

In [10]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()



except Exception as e:
    raise e

[2026-04-25 08:41:24,145: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-25 08:41:24,147: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-25 08:41:24,151: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-04-25 08:41:24,153: INFO: common: created directory at: artifacts]
[2026-04-25 08:41:24,155: INFO: common: created directory at: artifacts/model_trainer]


debug